# SFT on MBPP (Qwen2.5-Coder-3B, LoRA) – Windows Version

Supervised fine-tuning of **Qwen2.5-Coder-3B-Instruct** on the MBPP subset of `final_dataset_v2_train.csv` using **LoRA** (no quantization). Windows-compatible; GPU recommended (~6–8GB VRAM).

Outputs LoRA adapters in PEFT format and a FedAvg-ready `lora_state_dict.pt`.

In [2]:
# Check GPU (optional; skip if no NVIDIA GPU)
import subprocess
try:
    subprocess.run(["nvidia-smi"], check=True)
except Exception as e:
    print("nvidia-smi not available:", e)
print('check')

Sat Mar 14 10:18:40 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.288.01             Driver Version: 535.288.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA RTX A4000               Off | 00000000:55:00.0 Off |                  Off |
| 41%   36C    P8              12W / 140W |     77MiB / 16376MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [3]:
!pip install --no-cache-dir "transformers>=4.36" "peft>=0.7" "trl>=0.7,<0.20" "datasets" "accelerate" "pandas" "torch>=2.4" "torchvision" "torchaudio"

Defaulting to user installation because normal site-packages is not writeable


## Data loading

In [4]:
import os
import pandas as pd

# Local path: same directory as this notebook (or set CSV_PATH / NOTEBOOK_DIR if needed)
try:
    NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    NOTEBOOK_DIR = os.getcwd()
CSV_PATH = os.path.abspath(os.path.join(NOTEBOOK_DIR, "..", "final_dataset_v2_train.csv"))
print("Using CSV:", CSV_PATH)

df = pd.read_csv(CSV_PATH)
print("Total rows:", len(df))

Using CSV: /home/jovyan/FED/MBPP/final_dataset_v2.csv
Total rows: 1491


In [5]:
df_mbpp = df[df["dataset"] == "mbpp"].copy()
df_mbpp = df_mbpp.dropna(subset=["prompt", "canonical_solution"])
df_mbpp["canonical_solution"] = df_mbpp["canonical_solution"].astype(str).str.strip()
df_mbpp = df_mbpp[df_mbpp["canonical_solution"].str.len() > 0]
assert len(df_mbpp) > 0, f"Expected MBPP rows, got {len(df_mbpp)}"
print("MBPP rows:", len(df_mbpp))

MBPP rows: 120


## Parse prompt and build chat messages

In [6]:
SEP = "\n\nUser: "

def row_to_messages(row):
    prompt_str = str(row["prompt"]).strip()
    idx = prompt_str.find(SEP)
    if idx == -1:
        import warnings
        warnings.warn(f"Row {row.get('task_id')}: no '\\n\\nUser: ' found; using whole prompt as user.")
        system_content = "You are an expert Python developer. Complete the function provided by the user."
        user_content = prompt_str.replace("System: ", "", 1).strip()
    else:
        system_content = prompt_str[:idx].replace("System: ", "", 1).strip()
        user_content = prompt_str[idx + len(SEP):].strip()
    raw_status = str(row.get("status", "")).strip().lower()
    gen_code = str(row.get("generated_code", "")).strip()
    if raw_status == "passed" and len(gen_code) > 0:
        solution = gen_code
    else:
        solution = str(row["canonical_solution"]).strip()
    return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": solution},
    ]

messages_list = [row_to_messages(row) for _, row in df_mbpp.iterrows()]
print("Built", len(messages_list), "message lists.")

Built 120 message lists.


In [7]:
from datasets import Dataset

train_dataset = Dataset.from_dict({"messages": messages_list})
print(train_dataset)

Dataset({
    features: ['messages'],
    num_rows: 120
})


## Model and tokenizer

In [8]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token 
print("Tokenizer loaded.")

Tokenizer loaded.


In [9]:
import torch
from transformers import AutoModelForCausalLM

compute_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=compute_dtype,
    device_map="auto",
    trust_remote_code=True,
)
print("Model loaded (fp16/bf16, no quantization).")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model loaded (fp16/bf16, no quantization).


## LoRA (PEFT) configuration

**FedAvg**: Use this exact config on all clients so state dict keys and shapes match when averaging.

In [10]:
from peft import LoraConfig, get_peft_model

LORA_R = 16
LORA_ALPHA = 32
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

Skipping import of cpp extensions due to incompatible torch version 2.10.0+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


## Training

In [11]:
from trl import SFTTrainer, SFTConfig

OUTPUT_BASE = NOTEBOOK_DIR if "NOTEBOOK_DIR" in dir() else os.getcwd()
output_dir = os.path.join(OUTPUT_BASE, "sft_mbpp_output")
ADAPTER_DIR = os.path.join(output_dir, "lora_adapters")

# Completion-only loss: mask prompt tokens (try DataCollatorForCompletionOnlyLM if available)
try:
    from trl import DataCollatorForCompletionOnlyLM
    response_template = "<|im_start|>assistant\n"
    collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)
except ImportError:
    try:
        from trl.extras import DataCollatorForCompletionOnlyLM
        response_template = "<|im_start|>assistant\n"
        collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)
    except ImportError:
        collator = None

training_args = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_strategy="epoch",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    max_grad_norm=0.3,
    save_total_limit=1,
    max_seq_length=1024,
    packing=False,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [12]:
def formatting_func(example):
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    formatting_func=formatting_func,
    processing_class=tokenizer,
)
if collator is not None:
    trainer_kwargs["data_collator"] = collator

trainer = SFTTrainer(**trainer_kwargs)
print("SFTTrainer created.")
print(len(train_dataset))

Applying formatting function to train dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

SFTTrainer created.
120


In [13]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
5,0.890556
10,0.453437
15,0.356907
20,0.335029
25,0.278650
30,0.326187
35,0.310554
40,0.238383
45,0.189786


TrainOutput(global_step=45, training_loss=0.3754986868964301, metrics={'train_runtime': 108.21, 'train_samples_per_second': 3.327, 'train_steps_per_second': 0.416, 'total_flos': 881463419043840.0, 'train_loss': 0.3754986868964301})

## Save adapter (PEFT format)

In [15]:
import os
os.makedirs(ADAPTER_DIR, exist_ok=True)
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved adapter and tokenizer to", ADAPTER_DIR)
print("Files:", os.listdir(ADAPTER_DIR))

Saved adapter and tokenizer to /home/jovyan/FED/MBPP/sft_mbpp_output/lora_adapters
Files: ['adapter_model.safetensors', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json', 'training_args.bin', 'README.md', 'adapter_config.json']


## Get and print LoRA adapters (FedAvg-ready, visible as output)

In [16]:
lora_state = {
    k: v.detach().cpu().clone()
    for k, v in model.named_parameters()
    if v.requires_grad
}

print("=== LoRA adapter structure (name -> shape) ===")
for name, tensor in lora_state.items():
    print(f"  {name}: {tensor.shape}")

print("\n=== Per-parameter summary (min, max, norm) ===")
for name, tensor in lora_state.items():
    t = tensor.float()
    print(f"  {name}: min={t.min().item():.4f}, max={t.max().item():.4f}, norm={t.norm().item():.4f}")

total_params = sum(p.numel() for p in lora_state.values())
print(f"\n=== Summary ===")
print(f"  Number of LoRA parameters: {len(lora_state)}")
print(f"  Total elements: {total_params}")
print("  Parameter names (for FedAvg):", list(lora_state.keys()))

=== LoRA adapter structure (name -> shape) ===
  base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight: torch.Size([16, 2048])
  base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight: torch.Size([2048, 16])
  base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight: torch.Size([16, 2048])
  base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight: torch.Size([256, 16])
  base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight: torch.Size([16, 2048])
  base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight: torch.Size([256, 16])
  base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight: torch.Size([16, 2048])
  base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight: torch.Size([2048, 16])
  base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight: torch.Size([16, 2048])
  base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight: torch.Size([11

**FedAvg**: Use `lora_state_dict.pt` below for aggregation. Load it with `torch.load(...)` and average the tensors with other clients' LoRA state dicts (same keys and shapes).

In [17]:
lora_pt_path = os.path.join(ADAPTER_DIR, "lora_state_dict.pt")
torch.save(lora_state, lora_pt_path)
print("Saved FedAvg-ready LoRA state dict to", lora_pt_path)
print("File size (MB):", os.path.getsize(lora_pt_path) / (1024 * 1024))

# Outputs are in sft_mbpp_output/; copy that folder to backup if needed.


Saved FedAvg-ready LoRA state dict to /home/jovyan/FED/MBPP/sft_mbpp_output/lora_adapters/lora_state_dict.pt
File size (MB): 114.35906887054443
